In [3]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os 

In [4]:
import sys
sys.path.append("..")

In [5]:
from utils.iv_solver import iv_newton
from utils.get_market_data import get_candles

In [7]:
from datetime import datetime, timedelta
now = datetime.now()
from_ = now - timedelta(days=1)
sber_price = get_candles("SBER", from_, now)

2026-05-19 17:40:18.213045
Number of deleted duplicates: 0


In [8]:
sber_price.head(1)

,open,close,high,low,value,volume,end
timestamp,,,,,,,
2026-05-16 17:50:00,323.54,323.54,323.55,323.47,1825564.15,5643,2026-05-16 17:59:59


In [53]:
import numpy as np
from itertools import product

call_suff = ["CE6", "CE6P", "CQ6P"]
put_suff = ["CQ6", "CQ6D", "CR6A"]
expiry_dates = [datetime(2026, 5, 20), datetime(2026, 5, 27), datetime(26, 6, 3)]
strikes = np.linspace(270, 380, 12, dtype=int)

put_tickers  = [f"SR{s}{suf}" for s, suf in product(strikes, put_suff)]
call_tickers = [f"SR{s}{suf}" for s, suf in product(strikes, call_suff)]
len(call_tickers)

36

In [34]:
load_dotenv()
db_url = os.getenv("DB_URL")

In [64]:
import psycopg2

conn = psycopg2.connect(db_url)
cur = conn.cursor()

In [73]:
rows_arr = []
for ticker in call_tickers:
    print(ticker)
    cur.execute("""
    SELECT ticker, bids, asks FROM orderbooks
    WHERE ticker = %s
    ORDER BY timestamp ASC LIMIT 1
    """, (ticker,))
    rows = cur.fetchall()
    rows_arr.extend(rows)

SR270CE6
SR270CE6P
SR270CQ6P
SR280CE6
SR280CE6P
SR280CQ6P
SR290CE6
SR290CE6P
SR290CQ6P
SR300CE6
SR300CE6P
SR300CQ6P
SR310CE6
SR310CE6P
SR310CQ6P
SR320CE6
SR320CE6P
SR320CQ6P
SR330CE6
SR330CE6P
SR330CQ6P
SR340CE6
SR340CE6P
SR340CQ6P
SR350CE6
SR350CE6P
SR350CQ6P
SR360CE6
SR360CE6P
SR360CQ6P
SR370CE6
SR370CE6P
SR370CQ6P
SR380CE6
SR380CE6P
SR380CQ6P


In [74]:
df = pd.DataFrame(rows_arr, columns = ["ticker", "bids", "asks"])
df['best_bid'] = df['bids'].apply(lambda x: x[0]['price'] if x else None)
df['best_ask'] = df['asks'].apply(lambda x: x[0]['price'] if x else None)
df['mid'] = (df['best_ask'] + df['best_bid'])/2

In [75]:
df = df[["ticker", "mid"]]
df

,ticker,mid
0,SR270CE6,55.08
1,SR280CE6,45.40
2,SR290CE6,NaN
3,SR300CE6,NaN
4,SR310CE6,NaN
5,SR320CE6,NaN
6,SR330CE6,NaN
7,SR340CE6,NaN
8,SR350CE6,NaN
9,SR360CE6,NaN
